# BoldSearch Kaggle wrapper

Kaggle chỉ là máy chạy BE/FE như local. Attach vào Input: (1) dataset keyframes — slug phải khớp `KEYFRAMES_DIR` trong `.env`, (2) dataset private `.env` do CI publish, đã chứa sẵn path Kaggle (`KEYFRAMES_DIR` = `/kaggle/input/<slug>/keyframes`, `FRAME_IMAGE_URL_TEMPLATE` đúng đuôi file corpus).

Settings: GPU + Internet bật, rồi **Run All**. Repo tự validate (`AppConfig` parse env, lifespan kết nối Zilliz); wrapper chỉ: pull repo → chép env → chạy BE/FE → tunnel.

In [ ]:
import json
import os
import shutil
import subprocess
from pathlib import Path

runtime = Path("/kaggle/working/boldsearch-runtime")
runtime.mkdir(exist_ok=True)

print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip() or "no GPU — BEiT3 sẽ fail ở startup")

env_candidates = [e / ".env" for e in Path("/kaggle/input").iterdir() if (e / ".env").is_file()]
assert len(env_candidates) == 1, f"attach đúng 1 env dataset của CI, thấy: {env_candidates}"
env_file = env_candidates[0]


def env_value(key: str) -> str:
    for line in env_file.read_text(encoding="utf-8").splitlines():
        if line.startswith(key + "="):
            value = line.split("=", 1)[1].strip()
            return json.loads(value) if value.startswith('"') else value
    return ""


REPO_URL = env_value("REPO_URL")
assert REPO_URL, "env dataset thiếu REPO_URL"
repo = Path("/kaggle/working/BoldSearch")

askpass = runtime / "git-askpass.sh"
askpass.write_text(
    '#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *) echo "$BOLDSEARCH_GIT_TOKEN" ;; esac\n',
    encoding="utf-8",
)
askpass.chmod(0o700)
git_env = {**os.environ, "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0", "BOLDSEARCH_GIT_TOKEN": env_value("GH_PAT")}

if (repo / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin"], env=git_env, check=True)
    subprocess.run(["git", "-C", str(repo), "reset", "--hard", "FETCH_HEAD"], env=git_env, check=True)
else:
    if repo.exists():
        shutil.rmtree(repo)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(repo)], env=git_env, check=True)
askpass.unlink()

# env dataset = backend .env nguyên văn, đã chứa path Kaggle từ secret
shutil.copy(env_file, repo / "app" / "backend" / ".env")
print("commit:", subprocess.check_output(["git", "-C", str(repo), "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
import json
import subprocess
import time
from urllib.request import ProxyHandler, Request, build_opener

http = build_opener(ProxyHandler({}))  # health check cục bộ, bỏ proxy môi trường
backend = repo / "app" / "backend"

subprocess.run(
    ["uv", "sync", "--locked", "--no-dev"],
    cwd=backend, env={**os.environ, "UV_LINK_MODE": "copy"}, check=True,
)


def spawn(command, cwd, log_name):
    handle = (runtime / log_name).open("ab")
    return subprocess.Popen(command, cwd=cwd, stdout=handle, stderr=subprocess.STDOUT, start_new_session=True)


def wait_url(url, timeout=300):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            with http.open(url, timeout=5) as response:
                if response.status == 200:
                    return
        except Exception:
            time.sleep(2)
    raise RuntimeError(f"not ready after {timeout}s: {url}")


spawn(["uv", "run", "--no-sync", "uvicorn", "main:app", "--host", "127.0.0.1", "--port", "8000"], backend, "backend.log")
wait_url("http://127.0.0.1:8000/api/health")
print("BACKEND=ok")

frontend = repo / "app" / "frontend"
subprocess.run(["npm", "ci", "--no-audit", "--fund=false"], cwd=frontend, check=True)
spawn([str(frontend / "node_modules/.bin/vite"), "--host", "0.0.0.0", "--port", "5173"], frontend, "frontend.log")
base = "http://127.0.0.1:5173"
wait_url(base + "/", 120)
print("FRONTEND=ok")

smoke_request = Request(
    base + "/api/search/query",
    data=json.dumps({"query": env_value("SMOKE_QUERY") or "person", "task": "KIS", "topK": 1}).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with http.open(smoke_request, timeout=120) as response:
    results = json.loads(response.read().decode("utf-8"))["results"]
assert results, "smoke query không có kết quả — kiểm tra backend.log"
print("SMOKE=passed", results[0]["video_id"], results[0]["frame_id"])

In [ ]:
import re
import subprocess
import time
from urllib.request import urlretrieve

cloudflared = runtime / "cloudflared"
if not cloudflared.is_file():
    urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        cloudflared,
    )
    cloudflared.chmod(0o755)

tunnel_log = runtime / "tunnel.log"
with tunnel_log.open("ab") as handle:
    subprocess.Popen(
        [str(cloudflared), "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:5173"],
        stdout=handle, stderr=subprocess.STDOUT, start_new_session=True,
    )

for _ in range(60):
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", tunnel_log.read_text(errors="replace"))
    if match:
        print(f"TUNNEL_URL={match.group(0)}")
        print("Session giữ sống; tắt bằng Kaggle UI hoặc deploy version mới.")
        break
    time.sleep(2)
else:
    raise RuntimeError("tunnel không public URL kịp timeout")
while True:
    time.sleep(60)